# Data cleaning

**Missing data**:
- Iterative imputer (+/- indicator)
- KNN imputer (+/- indicator)

**Feature encoding**
- One-hot
- Target

## 1. Notebook setup

### 1.1. Imports

In [1]:
# Standard library
import json

# Third party
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, LabelEncoder, StandardScaler

### 1.2. Run configuration

In [2]:
SAMPLE   = 0.10    # Fraction of data to use for training
CV_FOLDS = 5       # Number of cross-validation folds

## 2. Data preparation

### 2.1. Data loading 

In [3]:
# Load raw data
train_df = pd.read_csv('https://media.githubusercontent.com/media/gperdrizet/fullstack-2605/refs/heads/main/data/student-health-risk-train.csv')
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       690088 non-null  int64  
 1   health_condition         690088 non-null  object 
 2   sleep_duration           614089 non-null  float64
 3   heart_rate               682255 non-null  float64
 4   bmi                      676190 non-null  float64
 5   calorie_expenditure      637235 non-null  float64
 6   step_count               676172 non-null  float64
 7   exercise_duration        683187 non-null  float64
 8   water_intake             646611 non-null  float64
 9   diet_type                683187 non-null  object 
 10  stress_level             607277 non-null  object 
 11  sleep_quality            631757 non-null  object 
 12  physical_activity_level  653467 non-null  object 
 13  smoking_alcohol          661506 non-null  object 
 14  gend

### 2.2. Metadata loading

In [4]:
# Load dataset metadata
with open("../data/schema.json", "r") as f:
    metadata = json.load(f)

# Extract feature lists
features = metadata['features']
categorical_features = metadata['categorical_features']
continuous_features = metadata['continuous_features']
high_cardinality_feature = metadata['high_cardinality_feature']
label = metadata['label']

# Treat 'step_count' like a continuous feature
continuous_features += [high_cardinality_feature]

print(f'Categorical features: {", ".join(categorical_features)}')
print(f'Continuous features: {", ".join(continuous_features)}')
print(f'Label: {label}')

Categorical features: diet_type, stress_level, sleep_quality, physical_activity_level, smoking_alcohol, gender
Continuous features: sleep_duration, heart_rate, bmi, calorie_expenditure, step_count, exercise_duration, water_intake, step_count
Label: health_condition


### 2.3. Data sampling

In [5]:
# Take sample
n = int(len(train_df) * SAMPLE)
train_df = train_df.sample(n=n)

### 2.4. Data cleaning and formatting

In [6]:
# Preserve label column
training_label = train_df['health_condition']

# Remove id and label columns
train_df.drop(['id', 'health_condition'], axis=1, inplace=True)
train_df.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
Index: 69008 entries, 13060 to 235919
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   sleep_duration           61364 non-null  float64
 1   heart_rate               68237 non-null  float64
 2   bmi                      67607 non-null  float64
 3   calorie_expenditure      63724 non-null  float64
 4   step_count               67614 non-null  float64
 5   exercise_duration        68291 non-null  float64
 6   water_intake             64720 non-null  float64
 7   diet_type                68292 non-null  object 
 8   stress_level             60650 non-null  object 
 9   sleep_quality            63262 non-null  object 
 10  physical_activity_level  65364 non-null  object 
 11  smoking_alcohol          66102 non-null  object 
 12  gender                   66856 non-null  object 
dtypes: float64(7), object(6)
memory usage: 7.4+ MB


## 3. Functions

### 3.1. Imputation

#### 3.1.1. Standard scaler

In [7]:
def standard_scaler(
        features: list[str],
        train_df: pd.DataFrame,
        test_df: pd.DataFrame = None
) -> tuple[pd.DataFrame, pd.DataFrame, StandardScaler] | tuple[pd.DataFrame, StandardScaler]:
    '''Standardize continuous features in the given DataFrames using Scikit-learn's StandardScaler.

    Args:
        features (list[str]): List of continuous features to standardize.
        train_df (pd.DataFrame): The training DataFrame.
        test_df (pd.DataFrame, optional): The testing DataFrame. Defaults to None.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame, StandardScaler] or tuple[pd.DataFrame, StandardScaler]:
        The training and testing dataframes with standardized continuous features and the fitted StandardScaler.
    '''

    scaler = StandardScaler()
    scaler.fit(train_df[features])

    train_df[features] = scaler.transform(train_df[features])

    if test_df is not None:
        test_df[features] = scaler.transform(test_df[features])

    if test_df is None:
        return train_df, scaler

    return train_df, test_df, scaler


#### 3.1.2. Missing indicator feature

In [8]:
def add_missing_indicator(
        features: list[str],
        train_df: pd.DataFrame,
        test_df: pd.DataFrame = None
) -> tuple[pd.DataFrame, pd.DataFrame] | pd.DataFrame:
    '''Add missing value indicator features to the given DataFrames.

    Args:
        features (list): List of features to create missing value indicators for.
        train_df (pd.DataFrame): The training DataFrame.
        test_df (pd.DataFrame): The testing DataFrame. Default is None.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame] or pd.DataFrame: The training and testing DataFrames with added missing value indicator features.
    '''

    # Create column names for the new indicator features
    indicator_features = [f'{feature}_missing' for feature in features]

    # Create indicator features from boolean mask
    training_missing = train_df[features].isna().astype('int')
    training_missing.columns = indicator_features

    # Add indicator features to DataFrames
    train_df = pd.concat(
        [train_df.reset_index(drop=True),
        training_missing.reset_index(drop=True)],
        axis=1
    )

    # Create indicator features for the test dataset if available
    if test_df is not None:
        testing_missing = test_df[features].isna().astype('int')
        testing_missing.columns = indicator_features

        test_df = pd.concat(
            [test_df.reset_index(drop=True),
            testing_missing.reset_index(drop=True)],
            axis=1
        )

    if test_df is None:
        return train_df

    return train_df, test_df

#### 3.1.3. KNN imputer

In [9]:
def knn_imputer(
        features: list[str],
        train_df: pd.DataFrame,
        test_df: pd.DataFrame = None,
        n_neighbors: int = 5,
        indicator: bool = False
) -> tuple[pd.DataFrame, pd.DataFrame] | pd.DataFrame:
    '''Impute missing values in the given features using K-Nearest Neighbors (KNN) imputer.

    Args:
        features (list[str]): The list of features to impute.
        train_df (pd.DataFrame): The training data with missing values.
        test_df (pd.DataFrame): The testing data with missing values.
        n_neighbors (int, optional): Number of neighbors to use for imputation. Defaults to 5.
        indicator (bool, optional): Whether to add a boolean indicator for missing values. Defaults to False.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame] or pd.DataFrame: The training and testing features with missing values imputed.
    '''

    # Scale the features before imputation
    if test_df is not None:
        train_df, test_df, scaler = standard_scaler(features, train_df, test_df)

    else:
        train_df, scaler = standard_scaler(features, train_df)

    # Create boolean indicator for missing values if requested
    if indicator:
        if test_df is not None:
            train_df, test_df = add_missing_indicator(features, train_df, test_df)

        else:
            train_df = add_missing_indicator(features, train_df)

    # Create the imputer
    imputer = KNNImputer(n_neighbors=n_neighbors, weights='distance')

    # Impute training dataset
    train_df = pd.DataFrame(
        imputer.fit_transform(train_df),
        columns=imputer.get_feature_names_out()
    )

    # Unscale the features after imputation
    train_df[features] = scaler.inverse_transform(train_df[features])

    # Impute test dataset if available
    if test_df is not None:
        test_df = pd.DataFrame(
            imputer.transform(test_df),
            columns=imputer.get_feature_names_out()
        )
        test_df[features] = scaler.inverse_transform(test_df[features])

    if test_df is None:
        return train_df

    return train_df, test_df

#### 3.1.4. Iterative imputer

In [10]:
def iterative_imputer(
        features: list[str],
        train_df: pd.DataFrame,
        test_df: pd.DataFrame = None,
        indicator: bool = False
) -> tuple[pd.DataFrame, pd.DataFrame] | pd.DataFrame:
    '''Impute missing values in the given features using Iterative Imputer.

    Args:
        features (list[str]): The list of features to impute.
        train_df (pd.DataFrame): The training data with missing values.
        test_df (pd.DataFrame, optional): The testing data with missing values. Defaults to None.
        indicator (bool, optional): Whether to add a boolean indicator for missing values. Defaults to False.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame] or pd.DataFrame: The training and testing dataframes with missing values imputed.
    '''

    # Scale the features before imputation
    if test_df is not None:
        train_df, test_df, scaler = standard_scaler(features, train_df, test_df)

    else:
        train_df, scaler = standard_scaler(features, train_df)

    # Create boolean indicator for missing values if requested
    if indicator:
        if test_df is not None:
            train_df, test_df = add_missing_indicator(features, train_df, test_df)

        else:
            train_df = add_missing_indicator(features, train_df)

    # Create the imputer
    imputer = IterativeImputer()

    # Impute training dataset
    train_df = pd.DataFrame(
        imputer.fit_transform(train_df),
        columns=imputer.get_feature_names_out()
    )

    # Unscale the features after imputation
    train_df[features] = scaler.inverse_transform(train_df[features])

    # Impute test dataset if available
    if test_df is not None:
        test_df = pd.DataFrame(
            imputer.transform(test_df),
            columns=imputer.get_feature_names_out()
        )
    
        test_df[features] = scaler.inverse_transform(test_df[features])

    if test_df is None:
        return train_df

    return train_df, test_df

### 3.2. Encoding

#### 3.2.1. One-hot encoder

In [11]:
def one_hot_encoder(
        features: list[str],
        train_df: pd.DataFrame,
        test_df: pd.DataFrame = None,
) -> tuple[pd.DataFrame, pd.DataFrame] | pd.DataFrame:
    '''Encodes features with Scikit-learn's one-hot-encoder.

    Args:
        features (list[str]): The list of features to encode.
        train_df (pd.DataFrame): The training data to encode.
        test_df (pd.DataFrame, optional): The testing data to encode. Default is None.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame] or pd.DataFrame: The training and testing (optional) dataframes with encoded features.
    '''
    
    # Scikit-learn one-hot encoder
    feature_encoder = OneHotEncoder(sparse_output=False, drop='first')

    # Adapt to training data categorical features
    feature_encoder.fit(train_df[features])

    # Transform training and testing data
    encoded_train_features = feature_encoder.transform(train_df[features])
    
    # Convert encoded features to DataFrame
    encoded_train_df = pd.DataFrame(encoded_train_features, columns=feature_encoder.get_feature_names_out())

    # Replace original categorical features with encoded features
    train_df = pd.concat(
        [train_df.drop(columns=features).reset_index(drop=True),
        encoded_train_df.reset_index(drop=True)],
        axis=1
    )

    # Encode test data if required
    if test_df is not None:

        encoded_test_features = feature_encoder.transform(test_df[features])
        encoded_test_df = pd.DataFrame(encoded_test_features, columns=feature_encoder.get_feature_names_out())

        test_df = pd.concat(
            [test_df.drop(columns=features).reset_index(drop=True),
            encoded_test_df.reset_index(drop=True)],
            axis=1
        )

    # Update feature list
    features = train_df.columns


    if test_df is None:
        return train_df
    
    return train_df, test_df
    

#### 3.2.2. Target encoder

In [12]:
def target_encoder(
        features: list[str],
        train_df: pd.DataFrame,
        train_label: pd.Series,
        test_df: pd.DataFrame = None,
        smooth: float | str = 'auto'
) -> tuple[pd.DataFrame, pd.DataFrame] | pd.DataFrame:
    '''Encodes features with Scikit-learn's target encoder.

    Args:
        features (list[str]): The list of features to encode.
        train_df (pd.DataFrame): The training data to encode.
        train_label (pd.Series): The training labels used to fit target encoding.
        test_df (pd.DataFrame, optional): The testing data to encode. Default is None.
        smooth (float|str, optional): Amount of mixing between category and global target mean. Default is 'auto'.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame] or pd.DataFrame: Encoded training and optional testing features.
    '''
    
    # Scikit-learn target encoder
    feature_encoder = TargetEncoder(smooth=smooth)

    # Adapt to training data categorical features
    feature_encoder.fit(train_df[features], train_label)

    # Transform training and testing data
    encoded_train_features = feature_encoder.transform(train_df[features])
    
    # Convert encoded features to DataFrame
    encoded_train_df = pd.DataFrame(encoded_train_features, columns=feature_encoder.get_feature_names_out())

    # Replace original categorical features with encoded features
    train_df = pd.concat(
        [train_df.drop(columns=features).reset_index(drop=True),
        encoded_train_df.reset_index(drop=True)],
        axis=1
    )

    # Encode test data if required
    if test_df is not None:
        encoded_test_features = feature_encoder.transform(test_df[features])
        encoded_test_df = pd.DataFrame(encoded_test_features, columns=feature_encoder.get_feature_names_out())

        test_df = pd.concat(
            [test_df.drop(columns=features).reset_index(drop=True),
            encoded_test_df.reset_index(drop=True)],
            axis=1
        )

    # Update feature list
    features = train_df.columns

    if test_df is None:
        return train_df

    return train_df, test_df

#### 3.2.3. Label encoder

In [13]:
def encode_label(label: pd.Series) -> tuple[pd.Series, LabelEncoder]:
    '''Encodes the training label with Scikit-learn's label encoder. Returns the encoded
    label as a series and the encoder.

    Args:
        label (pd.DataSeries): The training label column.

    Returns:
        tuple[pd.DataFrame, LabelEncoder]: The encoded labels and the LabelEncoder instance.
    '''

    
    # Scikit-learn label encoder
    label_encoder = LabelEncoder()

    # Convert string training labels to integer representation
    label = label_encoder.fit_transform(label)
    
    return label, label_encoder

### 4. Experiments

#### 4.1. Imputation x encoding strategy

##### 4.1.1. Condition definitions

In [14]:
conditions = {

    # One-hot encoding with KNN imputation (3, 5, 7 neighbors) - missing indicator
    'One-hot encoding, KNN imputation (3 neighbors), -indicator': {
        'Encoder': 'One-hot encoder',
        'Imputer': 'KNN imputer',
        'KNN neighbors': 3,
        'Missing indicator': False
    },
    'One-hot encoding, KNN imputation (5 neighbors), -indicator': {
        'Encoder': 'One-hot encoder',
        'Imputer': 'KNN imputer',
        'KNN neighbors': 5,
        'Missing indicator': False
    },
    'One-hot encoding, KNN imputation (7 neighbors), -indicator': {
        'Encoder': 'One-hot encoder',
        'Imputer': 'KNN imputer',
        'KNN neighbors': 7,
        'Missing indicator': False
    },

    # One-hot encoding with KNN imputation (3, 5, 7 neighbors) + missing indicator
    'One-hot encoding, KNN imputation (3 neighbors), +indicator': {
        'Encoder': 'One-hot encoder',
        'Imputer': 'KNN imputer',
        'KNN neighbors': 3,
        'Missing indicator': True
    },
    'One-hot encoding, KNN imputation (5 neighbors), +indicator': {
        'Encoder': 'One-hot encoder',
        'Imputer': 'KNN imputer',
        'KNN neighbors': 5,
        'Missing indicator': True
    },
    'One-hot encoding, KNN imputation (7 neighbors), +indicator': {
        'Encoder': 'One-hot encoder',
        'Imputer': 'KNN imputer',
        'KNN neighbors': 7,
        'Missing indicator': True
    },

    # Target encoding (smoothing 0.1) with KNN imputation (3, 5, 7 neighbors) - missing indicator
    'Target encoding (smoothing 0.1), KNN imputation (3 neighbors), -indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 0.1,
        'KNN neighbors': 3,
        'Missing indicator': False
    },
    'Target encoding (smoothing 0.1), KNN imputation (5 neighbors), -indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 0.1,
        'KNN neighbors': 5,
        'Missing indicator': False
    },
    'Target encoding (smoothing 0.1), KNN imputation (7 neighbors), -indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 0.1,
        'KNN neighbors': 7,
        'Missing indicator': False
    },

    # Target encoding (smoothing 1.0) with KNN imputation (3, 5, 7 neighbors) - missing indicator
    'Target encoding (smoothing 1.0), KNN imputation (3 neighbors), -indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 1.0,
        'KNN neighbors': 3,
        'Missing indicator': False
    },
    'Target encoding (smoothing 1.0), KNN imputation (5 neighbors), -indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 1.0,
        'KNN neighbors': 5,
        'Missing indicator': False
    },
    'Target encoding (smoothing 1.0), KNN imputation (7 neighbors), -indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 1.0,
        'KNN neighbors': 7,
        'Missing indicator': False
    },

    # Target encoding (smoothing 10.0) with KNN imputation (3, 5, 7 neighbors) - missing indicator
    'Target encoding (smoothing 10.0), KNN imputation (3 neighbors), -indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 10.0,
        'KNN neighbors': 3,
        'Missing indicator': False
    },
    'Target encoding (smoothing 10.0), KNN imputation (5 neighbors), -indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 10.0,
        'KNN neighbors': 5,
        'Missing indicator': False
    },
    'Target encoding (smoothing 10.0), KNN imputation (7 neighbors), -indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 10.0,
        'KNN neighbors': 7,
        'Missing indicator': False
    },

    # Target encoding (smoothing 0.1) with KNN imputation (3, 5, 7 neighbors) + missing indicator
    'Target encoding (smoothing 0.1), KNN imputation (3 neighbors), +indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 0.1,
        'KNN neighbors': 3,
        'Missing indicator': True
    },
    'Target encoding (smoothing 0.1), KNN imputation (5 neighbors), +indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 0.1,
        'KNN neighbors': 5,
        'Missing indicator': True
    },
    'Target encoding (smoothing 0.1), KNN imputation (7 neighbors), +indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 0.1,
        'KNN neighbors': 7,
        'Missing indicator': True
    },

    # Target encoding (smoothing 1.0) with KNN imputation (3, 5, 7 neighbors) + missing indicator
    'Target encoding (smoothing 1.0), KNN imputation (3 neighbors), +indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 1.0,
        'KNN neighbors': 3,
        'Missing indicator': True
    },
    'Target encoding (smoothing 1.0), KNN imputation (5 neighbors), +indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 1.0,
        'KNN neighbors': 5,
        'Missing indicator': True
    },
    'Target encoding (smoothing 1.0), KNN imputation (7 neighbors), +indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 1.0,
        'KNN neighbors': 7,
        'Missing indicator': True
    },

    # Target encoding (smoothing 10.0) with KNN imputation (3, 5, 7 neighbors) + missing indicator
    'Target encoding (smoothing 10.0), KNN imputation (3 neighbors), +indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 10.0,
        'KNN neighbors': 3,
        'Missing indicator': True
    },
    'Target encoding (smoothing 10.0), KNN imputation (5 neighbors), +indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 10.0,
        'KNN neighbors': 5,
        'Missing indicator': True
    },
    'Target encoding (smoothing 10.0), KNN imputation (7 neighbors), +indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'KNN imputer',
        'Target smoothing': 10.0,
        'KNN neighbors': 7,
        'Missing indicator': True
    },

    # Target encoding (smoothing 0.1, 1.0, 10.0) with iterative imputation - missing indicator
    'Target encoding (smoothing 0.1), iterative imputation, -indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'Iterative imputer',
        'Target smoothing': 0.1,
        'Missing indicator': False
    },
    'Target encoding (smoothing 1.0), iterative imputation, -indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'Iterative imputer',
        'Target smoothing': 1.0,
        'Missing indicator': False
    },
    'Target encoding (smoothing 10.0), iterative imputation, -indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'Iterative imputer',
        'Target smoothing': 10.0,
        'Missing indicator': False
    },

    # Target encoding (smoothing 0.1, 1.0, 10.0) with iterative imputation + missing indicator
    'Target encoding (smoothing 0.1), iterative imputation, +indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'Iterative imputer',
        'Target smoothing': 0.1,
        'Missing indicator': True
    },
    'Target encoding (smoothing 1.0), iterative imputation, +indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'Iterative imputer',
        'Target smoothing': 1.0,
        'Missing indicator': True
    },
    'Target encoding (smoothing 10.0), iterative imputation, +indicator': {
        'Encoder': 'Target encoder',
        'Imputer': 'Iterative imputer',
        'Target smoothing': 10.0,
        'Missing indicator': True
    },

    # One-hot encoding with iterative imputation - missing indicator
    'One-hot encoding, iterative imputation, -indicator': {
        'Encoder': 'One-hot encoder',
        'Imputer': 'Iterative imputer',
        'Missing indicator': False
    },

    # One-hot encoding with iterative imputation + missing indicator
    'One-hot encoding, iterative imputation, +indicator': {
        'Encoder': 'One-hot encoder',
        'Imputer': 'Iterative imputer',
        'Missing indicator': True
    }
}

print(f'Total {len(conditions.keys())} conditions')

Total 32 conditions


##### 4.1.2. Cross-validation

In [ ]:
results = {
    'Condition': [],
    'Cross-validation score': []
}

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=315)

for condition_name, condition_params in conditions.items():
    print(f'Evaluating condition: {condition_name}... ', end='')

    cv_scores = []

    # Fold-level preprocessing prevents train/validation leakage
    for train_idx, validation_idx in cv.split(train_df, training_label):
        x_train = train_df.iloc[train_idx].copy()
        x_validation = train_df.iloc[validation_idx].copy()

        y_train = training_label.iloc[train_idx]
        y_validation = training_label.iloc[validation_idx]

        # Apply the feature encoding on fold training data and transform validation data
        if condition_params['Encoder'] == 'One-hot encoder':
            x_train, x_validation = one_hot_encoder(
                features=categorical_features,
                train_df=x_train,
                test_df=x_validation
            )

        elif condition_params['Encoder'] == 'Target encoder':
            x_train, x_validation = target_encoder(
                features=categorical_features,
                train_df=x_train,
                train_label=y_train,
                test_df=x_validation,
                smooth=condition_params['Target smoothing']
            )

        # Apply imputation on fold training data and transform validation data
        if condition_params['Imputer'] == 'KNN imputer':
            x_train, x_validation = knn_imputer(
                features=continuous_features,
                train_df=x_train,
                test_df=x_validation,
                n_neighbors=condition_params['KNN neighbors'],
                indicator=condition_params['Missing indicator']
            )

        elif condition_params['Imputer'] == 'Iterative imputer':
            x_train, x_validation = iterative_imputer(
                features=continuous_features,
                train_df=x_train,
                test_df=x_validation,
                indicator=condition_params['Missing indicator']
            )

        model = GradientBoostingClassifier()
        model.fit(x_train, y_train)

        y_pred = model.predict(x_validation)
        score = balanced_accuracy_score(y_validation, y_pred)
        cv_scores.append(score)

    # Collect results
    results['Condition'].extend([condition_name] * CV_FOLDS)
    results['Cross-validation score'].extend(cv_scores)

    print(f'Balanced accuracy: {np.mean(cv_scores):.4f} +/- {np.std(cv_scores):.4f}')

Evaluating condition: One-hot encoding, KNN imputation (3 neighbors), -indicator... 